In [1]:
# Setup the Jupyter version of Dash
from dash import Dash, dcc, html, dash_table, Input, Output
import dash_leaflet as dl
import plotly.express as px
import base64

# Configure OS routines
import os

# Configure the plotting routines
import pandas as pd

# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter_crud import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"
host = "localhost"
port = 27017
db_name = "AAC"
collection_name = "animals"

# Connect to database via CRUD Module
db = AnimalShelter(username, password, host, port, db_name, collection_name)

def preprocess_animals(records):
    processed = {"all": [], "dogs": [], "cats": []}

    for r in records:
        if r is None:
            continue

        if "_id" in r:
            r.pop("_id", None)

        processed["all"].append(r)

        animal_type = r.get("animal_type")
        if animal_type == "Dog":
            processed["dogs"].append(r)
        elif animal_type == "Cat":
            processed["cats"].append(r)

    return processed

# Load once, preprocess once
raw_records = db.read({})
processed_records = preprocess_animals(raw_records)

# Initialize dataframe for the initial table view
df = pd.DataFrame.from_records(processed_records["all"])

#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

image_filename = "grazioso_logo.png"
encoded_image = base64.b64encode(open(image_filename, "rb").read())

app.layout = html.Div([
    html.Center([
        html.Img(
            src="data:image/png;base64,{}".format(encoded_image.decode()),
            style={"height": "140px"}
        ),
        html.H1("CS-340 Dashboard"),
        html.Div("Andrew Chacon")
    ]),
    html.Hr(),

    html.Div([
        dcc.Dropdown(
            id="filter-type",
            options=[
                {"label": "All", "value": "all"},
                {"label": "Dogs", "value": "dogs"},
                {"label": "Cats", "value": "cats"}
            ],
            value="all",
            clearable=False
        )
    ], style={"width": "300px"}),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0]
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        style={"display": "flex"},
        children=[
            html.Div(id="graph-id", style={"flex": "1"}),
            html.Div(id="map-id", style={"flex": "1"})
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_dashboard(filter_type):
    key = filter_type if filter_type in processed_records else "all"
    return processed_records[key]


@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graphs(viewData):
    if not viewData:
        return html.Div()

    dff = pd.DataFrame.from_dict(viewData)

    if "breed" in dff.columns and not dff["breed"].isna().all():
        fig = px.pie(dff, names="breed", title="Preferred Animals")
        return dcc.Graph(figure=fig)

    return html.Div()


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def update_styles(selected_columns):
    selected_columns = selected_columns or []
    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in selected_columns]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):
    if not viewData:
        return html.Div()

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return html.Div()

    row = index[0] if index else 0
    row = min(max(row, 0), len(dff) - 1)

    # Try to find latitude/longitude columns safely
    cols_lower = {c.lower(): c for c in dff.columns}
    lat_col = cols_lower.get("location_lat") or cols_lower.get("lat") or cols_lower.get("latitude")
    lon_col = cols_lower.get("location_long") or cols_lower.get("lon") or cols_lower.get("longitude") or cols_lower.get("long")

    if not lat_col or not lon_col:
        return html.Div("No location fields available for mapping.")

    lat = dff.iloc[row][lat_col]
    lon = dff.iloc[row][lon_col]

    if pd.isna(lat) or pd.isna(lon):
        return html.Div("Selected record has no coordinates.")

    breed_val = dff.iloc[row]["breed"] if "breed" in dff.columns else ""
    name_val = dff.iloc[row]["name"] if "name" in dff.columns else ""

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[float(lat), float(lon)],
                    children=[
                        dl.Tooltip(str(breed_val)),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(name_val))
                        ])
                    ]
                )
            ]
        )
    ]


app.run(debug=True)


Connected to MongoDB successfully.
